# 08. 用 Agent 写代码与排错

AGDR VLA 入门教程 · 第 3 章 · 后半周

- **学习目标**：掌握「理解 → 计划 → 小步修改 → 审查 diff → 验证 → 提交」的工作流程；能通过 `git diff` 审查 Agent 的修改，发现没有要求过的改动；能识别 Agent 排错时的危险信号
- **前置知识**：第 1 章（`git diff`、`git commit`、`git restore`）；[02-手写AgentLoop](./02-手写AgentLoop.ipynb)；[06-写好提示词](./06-写好提示词.ipynb)
- **预计学习时长**：1.5 小时

## 你是负责人，Agent 是执行者

用 Agent 写代码，最常见的错误是：Agent 说「改好了」，你就相信改好了。

Agent 写出的代码由**你**提交，出了问题也由你负责。它的工作很快，但它可能理解错需求、改动你没让它改的地方、用看起来合理的方式掩盖问题，而且总是语气自信地汇报。你的工作是**确认它做对了**。

推荐的工作流程：

```text
理解 ──▶ 计划 ──▶ 小步修改 ──▶ 审查 diff ──▶ 验证 ──▶ 提交
 │        │                      │            │         │
读懂相关  先出方案               逐行看       自己运行   一个改动
的代码    再动手（06 节）         改了什么     测试       一次提交
```

本节用一个小例子把这个流程完整走一遍。

## 准备：一个有 bug 的小项目

[data/buggy_project/](./data/buggy_project/) 下有一个很小的项目：`stats.py` 里的 `best_loss` 函数返回日志中最低的 loss，`test_stats.py` 是它的测试。**测试目前是失败的。**

先自己读一下这两个文件，看看你能不能发现 bug。然后运行下面的单元格：它把项目复制到沙箱工作区，初始化成一个 Git 仓库，并做一次初始提交，方便之后用 `git diff` 查看修改。

> 需要已经安装 Git（第 1 章）。Windows 用户如果没有 `python3` 命令，可以把后面录制的命令理解为 `python`；在线模式下模型会根据报错自己调整。

In [ ]:
import difflib
import shutil
import subprocess
from pathlib import Path

import mini_agent as ma

PROJECT_SRC = Path("data/buggy_project").resolve()
shutil.rmtree(ma.WORKSPACE, ignore_errors=True)
shutil.copytree(PROJECT_SRC, ma.WORKSPACE)


def git(*args):
    # 在沙箱工作区里运行一条 git 命令，并打印输出
    completed = subprocess.run(["git", *args], cwd=ma.WORKSPACE, capture_output=True, text=True)
    output = (completed.stdout + completed.stderr).rstrip()
    if output:
        print(output)


if shutil.which("git") is None:
    print("⚠️ 没有找到 git，后面查看 diff 的步骤无法运行。请先完成第 1 章的安装。")
else:
    git("init", "-q")
    git("add", ".")
    git("-c", "user.name=demo", "-c", "user.email=demo@example.com", "commit", "-q", "-m", "初始版本")
    git("log", "--oneline")
print("模式:", "在线" if ma.ONLINE else "离线（读取录制的模型响应）")

## 给 Agent 一个修改文件的工具

mini agent 目前只能读文件和运行命令。我们加一个 `write_file`，并且模仿真实 Agent 的做法：**写入之前先展示 diff，由你确认。**

`difflib` 是 Python 标准库，生成的 diff 格式和 `git diff` 一样：`-` 开头的行是删除的，`+` 开头的行是新增的。

In [ ]:
def write_file(path, content):
    target = ma.safe_path(path)
    old = target.read_text(encoding="utf-8") if target.exists() else ""
    diff = difflib.unified_diff(
        old.splitlines(keepends=True), content.splitlines(keepends=True),
        fromfile=f"a/{path}", tofile=f"b/{path}",
    )
    print(f"\n📝 Agent 想修改 {path}：\n")
    print("".join(diff))
    if input("允许写入吗？输入 y 写入，其他任意键拒绝：").strip().lower() != "y":
        return "用户拒绝了这次修改。"
    target.write_text(content, encoding="utf-8")
    return f"已写入 {path}"


ma.TOOL_SPECS[:] = [s for s in ma.TOOL_SPECS if s["function"]["name"] != "write_file"]
ma.TOOL_SPECS.append(ma.tool_spec(
    "write_file", "把内容写入工作区中的一个文件（整个文件覆盖）",
    {"path": {"type": "string", "description": "相对于工作区的文件路径"},
     "content": {"type": "string", "description": "文件的完整新内容"}},
    ["path", "content"],
))
ma.TOOLS["write_file"] = write_file
print("当前工具:", list(ma.TOOLS))

## 让 Agent 修 bug

运行下面的单元格，过程中会请你确认三次：运行测试、写入文件、再次运行测试。**每次确认前都读一读它要做什么。** 写入文件时会显示 diff，先别急着细看，后面会专门审查。

In [ ]:
answer, history = ma.run_agent(
    "test_stats.py 的测试没通过，帮我修好 stats.py。",
    offline_script=[
        "code_run_tests.json",  # 运行测试，看到失败
        "code_read.json",       # 读代码和测试
        "code_write.json",      # 修改 stats.py
        "code_rerun.json",      # 再次运行测试
        "code_answer.json",     # 汇报
    ],
)
print("\n最终回答:", answer)

Agent 的流程很规范：先复现失败，再读代码，找到原因，修改，再验证。它的解释也是对的：loss 被当作**字符串**比较了，按字符串比较时 `"12.4"` 排在 `"8.1"` 前面。这和第 2 章 `sort` 不加 `-n` 时 `10.5` 排错位置是同一个问题。

测试通过了，汇报也很清楚。可以提交了吗？

## 审查 diff

**不要看汇报，看 diff。** 汇报是 Agent 对自己工作的总结，diff 才是它实际做了什么。

In [ ]:
git("diff")

逐行对照，这次修改其实包含了**四处改动**：

| 改动 | 是否是你要求的 | 评价 |
| --- | --- | --- |
| `line.split(...)[1].strip()` → `float(... .split()[0])` | ✅ 这就是 bug 的修复 | 正确，而且只取 `loss=` 后面的第一个词，更稳健 |
| `for line in open(...)` → `with open(...) as f` | ❌ 没要求 | 无害，确保文件被关闭，算是顺手的小改进 |
| 新增 `if not losses: return None` | ❌ 没要求 | **改变了行为**：原来日志里没有 loss 时会报错，现在静默返回 `None`，调用它的代码可能因此出现更隐蔽的问题 |
| `min(losses)` → `round(min(losses), 2)` | ❌ 没要求 | **改变了结果**：loss 是 0.0012 时会返回 0.0。测试数据恰好都是两位小数，所以测试照样通过 |

而 Agent 的汇报里，**只提到了第一处**。

这就是审查 diff 的意义：测试通过只能说明测试覆盖到的情况是对的；汇报只说了 Agent 认为重要的部分。**没被要求的改动，尤其是改变了行为的改动，只能靠你读 diff 发现。**

审查时可以问自己：

- 每一处改动，我都知道它为什么要改吗？
- 有没有改动了我没提到的文件或函数？
- 有没有改变函数的输入、输出或报错方式？
- 有没有删除代码、注释或测试？
- 有没有新增依赖、硬编码的路径或密钥？

发现问题后，可以让 Agent 撤回不需要的改动，也可以自己动手改。动手试试第 5 题会让你自己完成这一步。

## 验证：不只是「测试通过」

Agent 运行了测试，你还要自己验证：

1. **自己运行一遍测试。** 不要只看 Agent 转述的结果
2. **想一想测试没覆盖的情况。** 这次的例子里：loss 很小时会怎样？日志为空时会怎样？loss 写成科学计数法 `1e-3` 时呢？
3. **读一遍最终的代码**，而不只是 diff。改动之后的整体逻辑还通顺吗？
4. 对于训练代码，「能跑」和「跑对」是两回事。可以先用很小的数据跑几步，看 loss 是否正常下降

## 小步提交

每完成一个经过验证的改动，就提交一次：

```bash
git diff                 # 审查
git add stats.py
git commit -m "修复 best_loss 把 loss 当字符串比较的问题"
```

小步提交的好处是：Agent 下一步改坏了，你可以用 `git restore` 轻松回到这个状态，也可以用 `git diff` 看清楚它在这之后又改了什么。如果让 Agent 连续工作一个小时、改了二十个文件才第一次审查，就很难理清了。

> 提交由你来做。本仓库的 `AGENTS.md` 规定 Agent 不能擅自提交，原因就在这里：**提交意味着你确认过这些改动。**

## 排错时的危险信号

Agent 很擅长让报错「消失」，但让报错消失不等于解决问题。看到下面这些做法时要警惕：

| 危险信号 | 为什么危险 |
| --- | --- |
| **修改测试**，让测试符合错误的代码 | 例如把测试里的 `8.1` 改成 `"12.4"`。测试是标准，不是障碍 |
| 用 `try: ... except: pass` 吞掉异常 | 报错不见了，问题还在，而且更难发现 |
| 删除或注释掉出错的代码 | 功能直接没了 |
| 加入特判，只针对测试数据返回正确结果 | 测试通过，换一组数据就错 |
| 降低检查标准，例如关掉类型检查、调大误差容限 | 同上 |
| 反复试错，每次都随便改一点 | 说明它没理解问题。应该打断它，让它先分析原因 |
| 说「已修复」，但没有运行任何验证 | 让它运行，并把结果贴出来 |

排错的正确顺序是：**复现 → 读懂报错 → 提出假设 → 用最小的实验验证假设 → 修复 → 确认测试覆盖了这个问题**。你可以在提示词里直接要求它按这个顺序来（06 节的「排错求助」模板）。

## 什么时候先别用 Agent

学习阶段要特别注意：

- **作业和练习**：目的是让你学会。可以让 Agent 讲解概念、分析报错、给提示，但代码要自己写。本仓库的学习者模式就是这么设计的。第 10 节作业一是例外：代码可以交给 Agent 写，但审查 diff 和提交不能交给它，这正是本节讲的流程
- **你完全看不懂它写的代码时**：先让它解释，或者自己查资料弄懂，再决定是否使用。你无法审查一段你看不懂的代码
- **涉及真机的代码**：控制机械臂的代码出错，后果可能是撞坏设备。这类代码必须逐行理解，并先在仿真或低速模式下测试

Agent 让你写代码更快，但不会替你理解代码。**先跑通、再理解、再改进**，理解这一步不能交给 Agent。

## 动手试试

不计分。第 1–4 题在离线模式下也能完成。

1. 不看上面的表格，自己读一遍 `git diff` 的输出，列出每一处改动，并判断它是不是修复 bug 所必需的。
2. 如果 Agent 的「修复」是把 `test_stats.py` 里的 `self.assertEqual(best_loss(f.name), 8.1)` 改成 `self.assertEqual(best_loss(f.name), "12.4")`，测试也会通过。这样做错在哪里？
3. 改写这次的任务提示词「test_stats.py 的测试没通过，帮我修好 stats.py。」，让 Agent 更不容易做出没要求的改动。
4. 写一个新的测试用例，能抓住 `round(..., 2)` 带来的问题。提示：什么样的 loss 会被 `round` 改变？
5. 在终端里进入沙箱工作区（运行 `print(ma.WORKSPACE)` 查看路径），**亲手**把 `stats.py` 里的 `round(..., 2)` 和 `if not losses: return None` 删掉，然后运行测试、用 `git diff` 检查、提交。

<details>
<summary>参考答案</summary>

**第 1 题**：见上文的表格。修复 bug 真正需要的只有把字符串转成 `float` 这一处。

**第 2 题**：测试描述的是**正确的行为**：最低的 loss 是 8.1。把期望值改成 `"12.4"`，等于宣布错误的结果是对的。测试通过了，bug 还在，而且以后再也不会被发现。遇到 Agent 修改测试时，一定要问清楚原因。

**第 3 题**（示例）：

```text
test_stats.py 的测试没通过。请找出 stats.py 中 best_loss 的 bug 并修复。
约束：只做修复 bug 所必需的最小改动，不要修改测试，不要改变函数的其他行为
（返回值类型、精度、出错时的表现）。
验收标准：python3 -m unittest test_stats 通过；在回答中列出你做的每一处改动和原因。
```

**第 4 题**：例如在 `LOG` 之外再测一份 loss 很小的日志：

```python
def test_small_loss(self):
    with tempfile.NamedTemporaryFile("w", suffix=".log", delete=False, encoding="utf-8") as f:
        f.write("[INFO] epoch=1 loss=0.0123\n[INFO] epoch=2 loss=0.0045\n")
    try:
        self.assertEqual(best_loss(f.name), 0.0045)
    finally:
        os.unlink(f.name)
```

`round(0.0045, 2)` 的结果是 `0.0`，这个测试会失败。

**第 5 题**：修改后 `best_loss` 的核心应该是：

```python
with open(path, encoding="utf-8") as f:
    for line in f:
        if "loss=" in line:
            losses.append(float(line.split("loss=")[1].split()[0]))
return min(losses)
```

然后依次运行：

```bash
python3 -m unittest test_stats
git diff
git add stats.py
git commit -m "修复 best_loss 把 loss 当字符串比较的问题"
```

`with open` 这一处保留与否都可以，它不改变行为。

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 说出「理解 → 计划 → 小步修改 → 审查 diff → 验证 → 提交」的流程 | 你是负责人 |
| 用 `git diff` 审查 Agent 的修改，找出没被要求的改动 | 审查 diff |
| 解释为什么「测试通过」和「Agent 说改好了」都不够 | 审查 diff、验证 |
| 说出小步提交的好处，以及为什么提交由你来做 | 小步提交 |
| 识别 Agent 排错时的危险信号 | 排错时的危险信号 |
| 判断什么时候应该先自己理解，而不是直接用 Agent | 什么时候先别用 Agent |

下一节：[09-安全风险与学术诚信](./09-安全风险与学术诚信.ipynb)